In [1]:
# Student-specific inputs
LAST_NAME = "Magalong"
STUDENT_ID = "TUPM-26-1134"
SEED_NUM = int(STUDENT_ID[-1])          # last digit of the student ID -> 4
FAVORITE_ARTIST = "TAYLOR SWIFT"        # <-- CHANGE THIS to your own favorite artist

# Character positions of FAVORITE_ARTIST (A=1, B=2, ... Z=26; spaces ignored)
ARTIST_POS = [ord(ch.upper()) - 64 for ch in FAVORITE_ARTIST if ch.isalpha()]

print("LAST_NAME      :", LAST_NAME)
print("SEED_NUM       :", SEED_NUM)
print("FAVORITE_ARTIST:", FAVORITE_ARTIST)
print("ARTIST_POS     :", ARTIST_POS)

LAST_NAME      : Magalong
SEED_NUM       : 4
FAVORITE_ARTIST: TAYLOR SWIFT
ARTIST_POS     : [20, 1, 25, 12, 15, 18, 19, 23, 9, 6, 20]


In [2]:
from collections import defaultdict, Counter

customers = [f"{LAST_NAME}_C{i}" for i in range(1, 6)]
products = ["NOTEBOOK", "PEN", "USB", "HEADPHONES", "CALCULATOR"]
prices = {"NOTEBOOK": 45, "PEN": 12, "USB": 350, "HEADPHONES": 899, "CALCULATOR": 650}   # fixed prices (PHP)

def make_qty(k):
    """Quantity 1-9 from SEED_NUM and artist character positions."""
    return (SEED_NUM * ARTIST_POS[k % len(ARTIST_POS)] + k) % 9 + 1

def txn_category(total):
    if total < 200: return "Small"
    if total < 1000: return "Medium"
    return "Large"

transactions = []
for t in range(12):
    cust = customers[t % 5]
    prod = products[(t * 2 + t // 5) % 5]
    transactions.append({"id": f"TX{t+1:02d}", "customer": cust, "product": prod, "qty": make_qty(t)})

# Repeated purchase: same customer-product pair as an earlier transaction
first = transactions[0]
transactions.append({"id": f"TX{len(transactions)+1:02d}", "customer": first["customer"],
                     "product": first["product"], "qty": make_qty(len(transactions))})

# Total value and category for each record
for tx in transactions:
    tx["price"] = prices[tx["product"]]
    tx["total"] = tx["qty"] * tx["price"]
    tx["category"] = txn_category(tx["total"])

print(f"{'ID':<6}{'CUSTOMER':<15}{'PRODUCT':<13}{'QTY':>4}{'PRICE':>7}{'TOTAL':>8}  CATEGORY")
for tx in transactions:
    print(f"{tx['id']:<6}{tx['customer']:<15}{tx['product']:<13}{tx['qty']:>4}{tx['price']:>7}{tx['total']:>8}  {tx['category']}")

ID    CUSTOMER       PRODUCT       QTY  PRICE   TOTAL  CATEGORY
TX01  Magalong_C1    NOTEBOOK        9     45     405  Medium
TX02  Magalong_C2    USB             6    350    2100  Large
TX03  Magalong_C3    CALCULATOR      4    650    2600  Large
TX04  Magalong_C4    PEN             7     12      84  Small
TX05  Magalong_C5    HEADPHONES      2    899    1798  Large
TX06  Magalong_C1    PEN             6     12      72  Small
TX07  Magalong_C2    HEADPHONES      2    899    1798  Large
TX08  Magalong_C3    NOTEBOOK        1     45      45  Small
TX09  Magalong_C4    USB             9    350    3150  Large
TX10  Magalong_C5    CALCULATOR      7    650    4550  Large
TX11  Magalong_C1    USB             1    350     350  Medium
TX12  Magalong_C2    CALCULATOR      2    650    1300  Large
TX13  Magalong_C1    NOTEBOOK        8     45     360  Medium


In [3]:
organized = defaultdict(lambda: defaultdict(list))
for tx in transactions:
    organized[tx["customer"]][tx["product"]].append((tx["id"], tx["qty"], tx["total"]))

for cust in sorted(organized):
    print(cust)
    for prod in sorted(organized[cust]):
        print(f"   {prod:<12}: {organized[cust][prod]}")

Magalong_C1
   NOTEBOOK    : [('TX01', 9, 405), ('TX13', 8, 360)]
   PEN         : [('TX06', 6, 72)]
   USB         : [('TX11', 1, 350)]
Magalong_C2
   CALCULATOR  : [('TX12', 2, 1300)]
   HEADPHONES  : [('TX07', 2, 1798)]
   USB         : [('TX02', 6, 2100)]
Magalong_C3
   CALCULATOR  : [('TX03', 4, 2600)]
   NOTEBOOK    : [('TX08', 1, 45)]
Magalong_C4
   PEN         : [('TX04', 7, 84)]
   USB         : [('TX09', 9, 3150)]
Magalong_C5
   CALCULATOR  : [('TX10', 7, 4550)]
   HEADPHONES  : [('TX05', 2, 1798)]


In [4]:
pair_counts = Counter((tx["customer"], tx["product"]) for tx in transactions)
repeated = {pair: n for pair, n in pair_counts.items() if n > 1}
distinct_products = {tx["product"] for tx in transactions}
distinct_customers = {tx["customer"] for tx in transactions}

overall_total = sum(tx["total"] for tx in transactions)
overall_qty = sum(tx["qty"] for tx in transactions)

print("Repeated customer-product purchases:")
for (c, p), n in repeated.items():
    print(f"  {c} bought {p} {n} times")
print("Distinct customers:", len(distinct_customers), sorted(distinct_customers))
print("Distinct products :", len(distinct_products), sorted(distinct_products))
print("\nOverall dataset value:", overall_total, "| total units:", overall_qty)

Repeated customer-product purchases:
  Magalong_C1 bought NOTEBOOK 2 times
Distinct customers: 5 ['Magalong_C1', 'Magalong_C2', 'Magalong_C3', 'Magalong_C4', 'Magalong_C5']
Distinct products : 5 ['CALCULATOR', 'HEADPHONES', 'NOTEBOOK', 'PEN', 'USB']

Overall dataset value: 18612 | total units: 64


In [5]:
customer_summary = {}
product_summary = {}
for tx in transactions:
    cs = customer_summary.setdefault(tx["customer"], {"transactions": 0, "units": 0, "spent": 0, "products": set()})
    cs["transactions"] += 1; cs["units"] += tx["qty"]; cs["spent"] += tx["total"]; cs["products"].add(tx["product"])
    ps = product_summary.setdefault(tx["product"], {"transactions": 0, "units": 0, "revenue": 0, "customers": set()})
    ps["transactions"] += 1; ps["units"] += tx["qty"]; ps["revenue"] += tx["total"]; ps["customers"].add(tx["customer"])

print("--- Customers ---")
for c, s in customer_summary.items():
    print(f"{c:<15}txns={s['transactions']} units={s['units']} spent={s['spent']} distinct_products={len(s['products'])}")
print("\n--- Products ---")
for p, s in product_summary.items():
    print(f"{p:<12}txns={s['transactions']} units={s['units']} revenue={s['revenue']} distinct_customers={len(s['customers'])}")

--- Customers ---
Magalong_C1    txns=4 units=24 spent=1187 distinct_products=3
Magalong_C2    txns=3 units=10 spent=5198 distinct_products=3
Magalong_C3    txns=2 units=5 spent=2645 distinct_products=2
Magalong_C4    txns=2 units=16 spent=3234 distinct_products=2
Magalong_C5    txns=2 units=9 spent=6348 distinct_products=2

--- Products ---
NOTEBOOK    txns=3 units=18 revenue=810 distinct_customers=2
USB         txns=3 units=16 revenue=5600 distinct_customers=3
CALCULATOR  txns=3 units=13 revenue=8450 distinct_customers=3
PEN         txns=2 units=13 revenue=156 distinct_customers=2
HEADPHONES  txns=2 units=4 revenue=3596 distinct_customers=2


In [6]:
customers_ranked = sorted(customer_summary.items(), key=lambda kv: kv[1]["spent"], reverse=True)
products_ranked = sorted(product_summary.items(), key=lambda kv: kv[1]["revenue"], reverse=True)
by_value = sorted(transactions, key=lambda tx: tx["total"], reverse=True)

print("Customers by total spending:")
for i, (c, s) in enumerate(customers_ranked, 1): print(f"  {i}. {c:<15}{s['spent']}")
print("\nProducts by revenue:")
for i, (p, s) in enumerate(products_ranked, 1): print(f"  {i}. {p:<12}{s['revenue']}")
print("\nTop 3 transactions by value:")
for tx in by_value[:3]: print(f"  {tx['id']}  {tx['customer']}  {tx['product']}  {tx['total']}")

Customers by total spending:
  1. Magalong_C5    6348
  2. Magalong_C2    5198
  3. Magalong_C4    3234
  4. Magalong_C3    2645
  5. Magalong_C1    1187

Products by revenue:
  1. CALCULATOR  8450
  2. USB         5600
  3. HEADPHONES  3596
  4. NOTEBOOK    810
  5. PEN         156

Top 3 transactions by value:
  TX10  Magalong_C5  CALCULATOR  4550
  TX09  Magalong_C4  USB  3150
  TX03  Magalong_C3  CALCULATOR  2600


In [7]:
line = "=" * 70
print(line)
print("CONSOLIDATED TRANSACTION REPORT".center(70))
print(f"Prepared by: {LAST_NAME} | ID: {STUDENT_ID} | Seed: {SEED_NUM}".center(70))
print(line)

print("TRANSACTIONS (ordered by value, highest first)")
print(f"{'ID':<6}{'CUSTOMER':<15}{'PRODUCT':<13}{'QTY':>4}{'TOTAL':>8}  CATEGORY")
for tx in by_value:
    print(f"{tx['id']:<6}{tx['customer']:<15}{tx['product']:<13}{tx['qty']:>4}{tx['total']:>8}  {tx['category']}")

print("\nCUSTOMER SUMMARY (by spending)")
print(f"{'CUSTOMER':<15}{'TXNS':>5}{'UNITS':>7}{'SPENT':>9}{'PRODUCTS':>10}")
for c, s in customers_ranked:
    print(f"{c:<15}{s['transactions']:>5}{s['units']:>7}{s['spent']:>9}{len(s['products']):>10}")

print("\nPRODUCT SUMMARY (by revenue)")
print(f"{'PRODUCT':<13}{'TXNS':>5}{'UNITS':>7}{'REVENUE':>9}{'CUSTOMERS':>11}")
for p, s in products_ranked:
    print(f"{p:<13}{s['transactions']:>5}{s['units']:>7}{s['revenue']:>9}{len(s['customers']):>11}")

print("\nREPEATED PURCHASES:", {f"{c}/{p}": n for (c, p), n in repeated.items()})
print("\nOVERALL TRANSACTION SUMMARY")
print(f"  Transactions   : {len(transactions)}")
print(f"  Total units    : {overall_qty}")
print(f"  Total value    : {overall_total}")
print(f"  Average value  : {round(overall_total / len(transactions), 2)}")
print(f"  Top customer   : {customers_ranked[0][0]} ({customers_ranked[0][1]['spent']})")
print(f"  Top product    : {products_ranked[0][0]} ({products_ranked[0][1]['revenue']})")
print(line)

                   CONSOLIDATED TRANSACTION REPORT                    
          Prepared by: Magalong | ID: TUPM-26-1134 | Seed: 4          
TRANSACTIONS (ordered by value, highest first)
ID    CUSTOMER       PRODUCT       QTY   TOTAL  CATEGORY
TX10  Magalong_C5    CALCULATOR      7    4550  Large
TX09  Magalong_C4    USB             9    3150  Large
TX03  Magalong_C3    CALCULATOR      4    2600  Large
TX02  Magalong_C2    USB             6    2100  Large
TX05  Magalong_C5    HEADPHONES      2    1798  Large
TX07  Magalong_C2    HEADPHONES      2    1798  Large
TX12  Magalong_C2    CALCULATOR      2    1300  Large
TX01  Magalong_C1    NOTEBOOK        9     405  Medium
TX13  Magalong_C1    NOTEBOOK        8     360  Medium
TX11  Magalong_C1    USB             1     350  Medium
TX04  Magalong_C4    PEN             7      84  Small
TX06  Magalong_C1    PEN             6      72  Small
TX08  Magalong_C3    NOTEBOOK        1      45  Small

CUSTOMER SUMMARY (by spending)
CUSTOMER        T